# Stage 5a — Assembling the commits

**Data engineering only.** No GPU, no model. This notebook reads the mined corpus, joins
everything a commit needs, and writes one table that Stage 5b classifies.

| in | out |
|---|---|
| Zenodo record 24182007 — diffs, refactorings, links, intervals, issues, pull requests | `stage5a_commits.jsonl` — one row per commit, pieces rendered, every join done |

Keeping the two apart means the joins can be inspected and corrected without re-running
classification, and classification can be re-run without re-downloading and re-joining.

## 0. Setup

Run this cell first. It works in Google Colab, Jupyter and IDEs such as VS Code or PyCharm.

It finds this repository (on Colab, opened straight from GitHub, it clones it), then picks one
working folder, `WORK`, where this notebook reads its inputs and writes its outputs. Every
notebook in the repository uses the same `WORK`, so they hand files to each other through it.

- set the environment variable `CIC_WORK` to choose the folder yourself
- otherwise it is `<repository>/work`, or on Colab a folder on your Google Drive
  (set `CIC_DRIVE=0` to stay on Colab's local disk)

An input missing from `WORK` is copied from the repository or downloaded from this study's Zenodo
record or Hugging Face repository, so any notebook can be run without re-running the ones before
it. See `replication.py`.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = "https://github.com/Binamra00/commit-intent-and-refactoring-levels"
_candidates = [Path(os.environ["CIC_ROOT"])] if os.environ.get("CIC_ROOT") else []
_candidates += [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in _candidates if (p / "replication.py").exists()), None)
if ROOT is None:                      # opened on its own, e.g. from GitHub in Colab
    ROOT = Path.cwd() / REPO.rsplit("/", 1)[-1]
    if not (ROOT / "replication.py").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)
sys.path.insert(0, str(ROOT))

from replication import setup, require
WORK = setup()

## 1. Inputs

In [1]:
import json, re, csv, zipfile, collections, urllib.request
from pathlib import Path
import pandas as pd

REPOS  = ["commons-lang", "checkstyle", "dubbo", "junit-framework", "questdb"]
RECORD = "24182007"   # the thesis corpus, public: doi.org/10.5281/zenodo.24182007

DRIVE = WORK          # the setup cell's working folder (on Drive when run in Colab)

DATA = Path("corpus"); DATA.mkdir(exist_ok=True)


def fetch_corpus():
    need = [r for r in REPOS if not any(DATA.rglob(f"diffs_{r}.jsonl"))]
    if not need:
        print("corpus already present"); return
    with urllib.request.urlopen(f"https://zenodo.org/api/records/{RECORD}", timeout=60) as r:
        meta = json.load(r)
    wanted = {f"{r}_mined.zip" for r in need}    # the record also holds git bundles; skip them
    for f in meta.get("files", []):
        name = f.get("key") or f.get("filename")
        if name not in wanted:
            continue
        dest = DATA / name
        if not dest.exists():
            url = f["links"].get("self") or f["links"].get("download")
            print(f"   {name} ({f.get('size', 0)/1e6:.0f} MB) ...", end=" ", flush=True)
            urllib.request.urlretrieve(url, dest); print("done")
        with zipfile.ZipFile(dest) as z:
            z.extractall(DATA / name[:-4])


def find(pattern):
    hits = sorted(DATA.rglob(pattern))
    return hits[0] if hits else None


def count_lines(path):
    if not path:
        return 0
    with open(path, encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())


fetch_corpus()

Mounted at /content/drive
Zenodo token (blank if public): ··········
   commons-lang.zip (26 MB) ... done
   checkstyle.zip (77 MB) ... done
   dubbo.zip (176 MB) ... done
   junit-framework.zip (64 MB) ... done
   questdb.zip (214 MB) ... done


In [2]:
# Counts, not presence. A column reading "yes" says a file exists; it says nothing about whether
# the file holds anything this stage can use.
FILES = [("diffs",        "diffs_{r}.jsonl",               True),
         ("refactorings", "refactorings_{r}.jsonl",        True),
         ("links",        "links_{r}.csv",                 True),
         ("intervals",    "intervals_{r}.json",            True),
         ("issues",       "channel_github_issues_{r}.jsonl", False),
         ("prs",          "channel_github_prs_{r}.jsonl",    False)]

rows, missing = [], []
for repo in REPOS:
    row = {"repo": repo}
    for name, pat, required in FILES:
        p = find(pat.format(r=repo))
        if name == "intervals":
            row[name] = (len(json.loads(p.read_text(encoding="utf-8"))["map"]) if p else 0)
        elif name == "links" and p:
            row[name] = count_lines(p) - 1            # header
        else:
            row[name] = count_lines(p)
        if required and not p:
            missing.append(pat.format(r=repo))
    rows.append(row)

print("RECORDS PER FILE\n")
print(pd.DataFrame(rows).set_index("repo").to_string())
print("""
diffs, refactorings   one record per Java commit in the universe
links                 edges from discourse to commits
intervals             commits placed in a release interval by ancestry
issues, prs           mined discourse records -- where maintainer labels come from""")
if missing:
    raise SystemExit(f"\nmissing required inputs: {missing}")

RECORDS PER FILE

                 diffs  refactorings   links  intervals  issues    prs
repo                                                                  
commons-lang      6914          6914   11301       9084       0   1703
checkstyle       10245         10245  254718      17053    5586  14512
dubbo             6992          6992   63276       8199    6588   8409
junit-framework   6181          6181   44108       8315    2551   2994
questdb           4741          4741   34146       6233    1636   4740

diffs, refactorings   one record per Java commit in the universe
links                 edges from discourse to commits
intervals             commits placed in a release interval by ancestry
issues, prs           mined discourse records -- where maintainer labels come from


## 2. Where the labels come from

Maintainer labels are **metadata**, not ground truth: Herzig, Just & Zeller (ICSE 2013) found
roughly a third of bug-labelled reports are not bugs. They are used here in two ways — reported
alongside each commit, and in Stage 5b as a noisy check on whether the classifier's confidence
means anything on this corpus.

A label reaches a commit through linkage: the commit is linked to an issue or pull request, and
that record carries labels. Both channels can supply them, and they are counted separately
because projects differ in which one they use.

In [3]:
def label_names(rec):
    return [l.get("name") if isinstance(l, dict) else str(l)
            for l in (rec.get("labels") or []) if l]


audit, VOCAB = [], {}
for repo in REPOS:
    for chan, pat in (("issues", "channel_github_issues_{r}.jsonl"),
                      ("prs", "channel_github_prs_{r}.jsonl")):
        p = find(pat.format(r=repo))
        n = labelled = 0
        vocab = collections.Counter()
        if p:
            with open(p, encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    rec = json.loads(line)
                    n += 1
                    names = label_names(rec)
                    if names:
                        labelled += 1
                        vocab.update(names)
        VOCAB[(repo, chan)] = vocab
        audit.append({"repo": repo, "channel": chan, "records": n, "labelled": labelled,
                      "%": round(labelled / n * 100, 1) if n else 0,
                      "distinct labels": len(vocab)})

A = pd.DataFrame(audit)
print("LABEL COVERAGE BY CHANNEL\n")
print(A.to_string(index=False))

print("\n\nMOST FREQUENT LABELS PER REPOSITORY\n")
for repo in REPOS:
    merged = VOCAB[(repo, "issues")] + VOCAB[(repo, "prs")]
    top = ", ".join(f"{k} ({v:,})" for k, v in merged.most_common(8)) or "none"
    print(f"   {repo:<18}{top}")

print("""
A repository with no labels at all -- commons-lang, which tracks its work in Apache JIRA rather
than GitHub -- contributes nothing to the Stage 5b check. That is a coverage limit, not a defect.""")

LABEL COVERAGE BY CHANNEL

           repo channel  records  labelled    %  distinct labels
   commons-lang  issues        0         0  0.0                0
   commons-lang     prs     1703       420 24.7                3
     checkstyle  issues     5586      4480 80.2               45
     checkstyle     prs    14512      1911 13.2               22
          dubbo  issues     6588      2025 30.7               16
          dubbo     prs     8409      1596 19.0               18
junit-framework  issues     2551      2379 93.3               51
junit-framework     prs     2994      1050 35.1               47
        questdb  issues     1636      1123 68.6               41
        questdb     prs     4740      1727 36.4               59


MOST FREQUENT LABELS PER REPOSITORY

   commons-lang      dependencies (420), github_actions (224), java (196)
   checkstyle        approved (4,373), miscellaneous (2,589), dependencies (1,170), bug (861), java (651), good first issue (308), awaiting chang

## 3. From labels to Swanson classes

Only labels whose meaning is unambiguous are mapped. Most labels in this corpus describe
**process** — `approved`, `dependencies`, `miscellaneous` — rather than the kind of work, and
guessing at those would inject more noise into a reference that is already noisy.

Levin's definitions, which the classifier was trained on:

| class | meaning |
|---|---|
| corrective | fixing faults |
| adaptive | adding new features |
| perfective | improving the system and its design |

A commit whose labels map to more than one class is **ambiguous** and left out of the check
rather than resolved by a rule.

In [4]:
LABEL_CLASS = {
    "corrective": {"bug", "defect", "regression", "crash", "fix", "bugfix", "bug fix"},
    "adaptive":   {"feature", "new feature", "enhancement", "improvement", "feature request"},
    "perfective": {"refactor", "refactoring", "cleanup", "clean up", "technical debt",
                   "tech debt", "code quality", "performance", "code style"},
}
LOOKUP = {v: k for k, vs in LABEL_CLASS.items() for v in vs}


def normalise(label):
    """'type/bug', 'type: bug', 'kind/bug' and 'Bug' all become 'bug'."""
    s = str(label).strip().lower()
    for sep in ("/", ":"):
        if sep in s:
            s = s.split(sep, 1)[1].strip()
    return s


def label_class(labels):
    """The single Swanson class the labels imply, 'ambiguous', or None."""
    classes = {LOOKUP[normalise(l)] for l in labels if normalise(l) in LOOKUP}
    if not classes:
        return None
    return classes.pop() if len(classes) == 1 else "ambiguous"


print("WHICH OBSERVED LABELS MAP, AND TO WHAT\n")
seen = collections.Counter()
for v in VOCAB.values():
    seen.update(v)
mapped = {l: LOOKUP[normalise(l)] for l in seen if normalise(l) in LOOKUP}
for cls in LABEL_CLASS:
    names = sorted((l for l, c in mapped.items() if c == cls), key=lambda l: -seen[l])
    total = sum(seen[l] for l in names)
    print(f"   {cls:<12}{total:>7,}   " + ", ".join(names[:8]))
unmapped = sum(v for l, v in seen.items() if l not in mapped)
print(f"\n   {unmapped:,} label occurrences carry no Swanson meaning and are ignored")
print(f"   top unmapped: " + ", ".join(l for l, _ in
      sorted(((l, v) for l, v in seen.items() if l not in mapped), key=lambda x: -x[1])[:8]))

WHICH OBSERVED LABELS MAP, AND TO WHAT

   corrective    3,457   Bug, type/bug, bug, type: bug, type: regression, regression, crash
   adaptive      2,231   type: enhancement, New feature, Enhancement, new feature, type: new feature, type/enhancement
   perfective      319   Performance

   27,408 label occurrences carry no Swanson meaning and are ignored
   top unmapped: approved, miscellaneous, dependencies, component: Jupiter, SQL, java, component: Platform, theme: programming model


## 4. Assemble the commits

One record per commit, rendered as the pieces the model was trained on: one for the message, one
per changed file, largest first, capped at Stage 4b's `MAX_PIECES`.

**Merges are dropped.** Their message is generated boilerplate and their first-parent diff is the
whole merged branch — neither describes work a developer intended.

In [5]:
cfg = require("stage4b_results.json", optional=True)
if cfg is None:
    raise SystemExit("stage4b_results.json not found -- it carries the MAX_PIECES the model was "
                     "trained with")
MAX_PIECES = json.load(open(cfg))["config"]["max_pieces"]
print(f"MAX_PIECES {MAX_PIECES}  (Stage 4b)\n")


def render_pieces(rec, max_pieces=MAX_PIECES):
    """Identical to Stage 4b's input: blank lines dropped, files largest first."""
    out = [f"<COMMIT_MESSAGE>{(rec.get('message') or '').strip()}</COMMIT_MESSAGE>"]
    files = []
    for f in rec.get("files") or []:
        removed = [l for l in (f.get("removed") or []) if l.strip()]
        added = [l for l in (f.get("added") or []) if l.strip()]
        if not removed and not added:
            continue
        part = [f"<FILE>{f.get('file')}</FILE>"]
        if removed:
            part.append("<REMOVED>" + "\n".join(removed) + "</REMOVED>")
        if added:
            part.append("<ADDED>" + "\n".join(added) + "</ADDED>")
        files.append("\n".join(part))
    files.sort(key=len, reverse=True)
    return out + files[:max_pieces - 1]


rows, dropped = [], collections.Counter()
for repo in REPOS:
    with open(find(f"diffs_{repo}.jsonl"), encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            if rec.get("status") != "success":
                dropped["not successful"] += 1; continue
            if rec.get("is_merge"):
                dropped["merge"] += 1; continue
            rows.append({"repo": repo, "sha": rec["sha1"], "timestamp": rec.get("timestamp"),
                         "pieces": render_pieces(rec), "n_files": rec.get("n_files", 0),
                         "n_pieces_uncapped": 1 + len(rec.get("files") or [])})

C = pd.DataFrame(rows)
print(C.groupby("repo").size().rename("commits").to_string())
print(f"\n{len(C):,} commits")
print("dropped: " + ", ".join(f"{k} {v:,}" for k, v in dropped.items()))
capped = int((C.n_pieces_uncapped > MAX_PIECES).sum())
print(f"capped at {MAX_PIECES} pieces: {capped:,} ({capped/len(C)*100:.2f}%)")

MAX_PIECES 256  (Stage 4b)

repo
checkstyle         10240
commons-lang        6818
dubbo               6681
junit-framework     6142
questdb             4728

34,609 commits
dropped: merge 464
capped at 256 pieces: 161 (0.47%)


## 5. Join

Refactorings and their Murphy-Hill levels, channel evidence, maintainer labels and the Swanson
class they imply, and the release interval.

Membership is computed from the files rather than read from linkage's `in_refm` / `in_ck`
flags, which are blank where a stage had not run when linkage executed.

In [6]:
LEVELS_FILE = require("refactoring_levels.yaml", optional=True)
if LEVELS_FILE:
    import yaml
    LEVELS = {r["name"].strip().lower(): r["level"]
              for r in yaml.safe_load(open(LEVELS_FILE, encoding="utf-8"))["refactorings"]}
    print(f"levels: {LEVELS_FILE.name}, {len(LEVELS)} types\n")
else:
    LEVELS = {}
    print("refactoring_levels.yaml not found -- level columns will be zero\n")


def refactorings_for(repo):
    out = {}
    with open(find(f"refactorings_{repo}.jsonl"), encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            refs = rec.get("refactorings") or []
            if refs:
                types = [r.get("type") for r in refs if r.get("type")]
                lv = collections.Counter(LEVELS.get((t or "").strip().lower()) for t in types)
                out[rec["sha1"]] = (types, lv)
    return out


def evidence_for(repo):
    routes, sources = collections.defaultdict(set), collections.defaultdict(set)
    p = find(f"links_{repo}.csv")
    if p:
        with open(p, encoding="utf-8", newline="") as f:
            for r in csv.DictReader(f):
                sha = r.get("target_sha")
                if sha:
                    routes[sha].add(r.get("route", ""))
                    sources[sha].add((r.get("source_channel", ""), str(r.get("source_id", ""))))
    return routes, sources


def labels_for(repo):
    """(channel, id) -> label names. Both key parts are strings, so the join cannot miss on type."""
    out = {}
    for chan, pat in (("github.issue", "channel_github_issues_{r}.jsonl"),
                      ("github.pr", "channel_github_prs_{r}.jsonl")):
        p = find(pat.format(r=repo))
        if not p:
            continue
        with open(p, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    rec = json.loads(line)
                    names = label_names(rec)
                    if names:
                        out[(chan, str(rec.get("record_id")))] = names
    return out


parts, report = [], []
for repo in REPOS:
    sub = C[C.repo == repo].copy()
    refs = refactorings_for(repo)
    routes, sources = evidence_for(repo)
    labels = labels_for(repo)
    iv = json.loads(find(f"intervals_{repo}.json").read_text(encoding="utf-8"))["map"]

    sub["has_refactoring"] = sub.sha.isin(refs)
    sub["n_refactorings"] = sub.sha.map(lambda s: len(refs[s][0]) if s in refs else 0)
    sub["refactoring_types"] = sub.sha.map(
        lambda s: "|".join(sorted(set(refs[s][0]))) if s in refs else "")
    for lvl in ("High", "Medium", "Low"):
        sub[f"n_{lvl.lower()}"] = sub.sha.map(
            lambda s, L=lvl: refs[s][1].get(L, 0) if s in refs else 0)

    sub["has_evidence"] = sub.sha.isin(routes)
    sub["evidence_routes"] = sub.sha.map(lambda s: "|".join(sorted(routes.get(s, ()))))
    lab = sub.sha.map(lambda s: sorted({l for src in sources.get(s, ())
                                        for l in labels.get(src, ())}))
    sub["maintainer_labels"] = lab.map("|".join)
    sub["label_class"] = lab.map(label_class)
    sub["release_interval"] = sub.sha.map(lambda s: iv.get(s, ""))
    parts.append(sub)
    report.append({"repo": repo,
                   "refactoring": int(sub.has_refactoring.sum()),
                   "evidence": int(sub.has_evidence.sum()),
                   "labelled": int((sub.maintainer_labels != "").sum()),
                   "label→class": int(sub.label_class.isin(LABEL_CLASS).sum()),
                   "ambiguous": int((sub.label_class == "ambiguous").sum()),
                   "in interval": int((sub.release_interval != "").sum())})

C = pd.concat(parts, ignore_index=True)
print(pd.DataFrame(report).set_index("repo").to_string())
print("""
label→class  commits whose labels imply exactly one Swanson class -- the Stage 5b reference
ambiguous    labels implying more than one class; excluded from the check, not resolved""")

levels: refactoring_levels.yaml, 107 types

                 refactoring  evidence  labelled  label→class  ambiguous  in interval
repo                                                                                 
commons-lang            1974       665         0            0          0         6314
checkstyle              3827      8108      7269         1181          5        10135
dubbo                   3164      4515       666          339          3         5463
junit-framework         3507      2837      2491         1647         58         3485
questdb                 3280      3086      1742         1255        132         4645

label→class  commits whose labels imply exactly one Swanson class -- the Stage 5b reference
ambiguous    labels implying more than one class; excluded from the check, not resolved


## 6. Write

In [7]:
OUT = DRIVE / "stage5a_commits.jsonl"
cols = ["repo", "sha", "timestamp", "release_interval", "pieces", "n_files", "n_pieces_uncapped",
        "has_refactoring", "n_refactorings", "n_high", "n_medium", "n_low", "refactoring_types",
        "has_evidence", "evidence_routes", "maintainer_labels", "label_class"]
with open(OUT, "w", encoding="utf-8") as f:
    for rec in C[cols].to_dict("records"):
        f.write(json.dumps(rec, ensure_ascii=False, default=str) + "\n")

json.dump({"commits": int(len(C)), "max_pieces": MAX_PIECES,
           "dropped": dict(dropped), "zenodo_record": RECORD,
           "label_class_map": {k: sorted(v) for k, v in LABEL_CLASS.items()}},
          open(DRIVE / "stage5a_meta.json", "w"), indent=1)

print(f"{OUT}   {len(C):,} commits, {OUT.stat().st_size/1e6:.0f} MB")
print(f"{DRIVE / 'stage5a_meta.json'}")
print("\nStage 5b reads this file and nothing else from the corpus.")

/content/drive/MyDrive/levin_stage4b/stage5a_commits.jsonl   34,609 commits, 994 MB
/content/drive/MyDrive/levin_stage4b/stage5a_meta.json

Stage 5b reads this file and nothing else from the corpus.
